# 4.03 Grid Search de Arboles Azarosos — Opcion B: validacion local

Recorre la grilla de hiperparametros < feature_fraction, minsplit, minbucket, maxdepth > (con **cp fijo en -1**), entrenando para cada combinacion el ensemble arbol por arbol (igual que en `z420`), separando el 202107 en train/validacion (70/30, particion estratificada) y midiendo la ganancia localmente en cada punto de `PARAM$grabar` (por default `1, 2, 4, 8, 16, 32`), con la misma formula de `z290_TareaHogar_02` (estimulo 975000 / costo -25000, corte en prob > 0.025).

<br>No consume submits de Kaggle, asi que esta grilla puede ser mucho mas grande que la de la Opcion A — el costo es tiempo de computo.

<br>Al final se muestra un ranking de combinaciones por la ganancia del ensemble completo (ultimo punto de `PARAM$grabar`). Elegis la mejor y la cargas manualmente en `z420` (los campos `PARAM$feature_fraction` y `PARAM$rpart`) para el entrenamiento final y el submit a Kaggle.

#### Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/utn2026-b40a/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"

---

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Fri Aug 21 03:39:18 PM 2026"

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671290,35.9,1473300,78.7,1473300,78.7
Vcells,1242666,9.5,8388608,64.0,1978711,15.1


In [ ]:
# cargo las librerias que necesito
require("data.table")
require("rpart")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: rpart



Aqui debe cargar SU semilla primigenia, y ajustar la grilla de hiperparametros a explorar

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 346321
PARAM$training_pct <- 70L # split train/validacion DENTRO de 202107, no toca Kaggle

PARAM$rpart$cp <- -1 # fijo, no se barre en este grid search

PARAM$num_trees_max <- 32 # arboles por combinacion, igual que z420

# puntos del ensemble donde se mide la ganancia, igual que en z420
PARAM$grabar <- c(1, 2, 4, 8, 16, 32)

# grilla de hiperparametros a explorar
#  al ser validacion local (no consume submits de Kaggle) esta grilla puede ser
#  mucho mas grande que la de la Opcion A, el unico costo es tiempo de computo
PARAM$grid$feature_fraction <- c(0.5, 0.7)
PARAM$grid$minsplit  <- c(1000, 800, 600, 400, 200, 100, 50, 20, 10)
PARAM$grid$minbucket <- c(5, 10, 20, 50, 100)
PARAM$grid$maxdepth  <- c(4, 6, 8, 10, 12, 14)

In [ ]:
PARAM

$semilla_primigenia
[1] 346321

$training_pct
[1] 70

$rpart
$rpart$cp
[1] -1


$num_trees_max
[1] 8

$grabar
[1] 1 2 4 8

$grid
$grid$feature_fraction
[1] 0.5 0.7

$grid$minsplit
[1] 100  50

$grid$minbucket
[1]  50 100 150

$grid$maxdepth
[1] 6 8

In [ ]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp4221"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

# me quedo solo con los datos que tienen clase conocida, es decir 202107
#  (202109 tiene clase_ternaria vacia, no sirve para validar localmente)
dataset <- dataset[clase_ternaria != ""]

### Particion train / validacion local (70/30 estratificada por clase_ternaria)

In [ ]:
# particionar agrega una columna llamada fold a un dataset
#  que consiste en una particion estratificada segun agrupa
particionar <- function(data, division, agrupa = "", campo = "fold", start = 1, seed = NA) {
  if (!is.na(seed)) set.seed(seed)

  bloque <- unlist(mapply(function(x, y) {
    rep(y, x)
  }, division, seq(from = start, length.out = length(division))))

  data[, (campo) := sample(rep(bloque, ceiling(.N / length(bloque))))[1:.N],
    by = agrupa
  ]
}

# hago la particion UNA sola vez, con semilla fija, para que todas las
#  combinaciones de la grilla se comparen sobre el mismo train/validation
particionar(dataset,
  division= c(PARAM$training_pct, 100L - PARAM$training_pct),
  agrupa= "clase_ternaria",
  seed= PARAM$semilla_primigenia
)

dtr  <- dataset[fold == 1] # 70% training
dval <- dataset[fold == 2] # 30% validacion local

campos_buenos <- copy(setdiff(colnames(dtr), c("clase_ternaria", "fold")))

### Funcion que entrena el ensemble y devuelve la ganancia en validacion

In [ ]:
# entrena el ensemble de PARAM$num_trees_max arboles para una combinacion de
#  hiperparametros dada, y devuelve la ganancia normalizada sobre dval en cada
#  punto de PARAM$grabar (una fila por punto, columnas arbolito y ganancia)
ArbolesAzarososGanancia <- function(feature_fraction, rpart_control) {

  # misma semilla para todas las combinaciones, asi la unica diferencia entre
  #  combinaciones es el hiperparametro, no el azar
  set.seed(PARAM$semilla_primigenia)

  tb_pred <- dval[, list(numero_de_cliente, clase_ternaria)]
  tb_pred[, prob_acumulada := 0]

  resultados <- data.table(arbolito = integer(), ganancia = numeric())

  for (arbolito in seq(PARAM$num_trees_max)) {
    qty_campos_a_utilizar <- as.integer(length(campos_buenos) * feature_fraction)
    campos_random <- sample(campos_buenos, qty_campos_a_utilizar)
    campos_random <- paste(campos_random, collapse= " + ")
    formulita <- paste0("clase_ternaria ~ ", campos_random)

    modelo <- rpart(formulita, data= dtr, xval= 0, control= rpart_control)
    prediccion <- predict(modelo, dval, type= "prob")
    tb_pred[, prob_acumulada := prob_acumulada + prediccion[, "BAJA+2"]]

    if (!(arbolito %in% PARAM$grabar)) next

    # umbral sobre la SUMA acumulada, equivalente a promedio > 1/40
    umbral_corte <- arbolito / 40
    tb_pred[, Predicted := prob_acumulada > umbral_corte]

    ganancia_test <- tb_pred[, sum(ifelse(Predicted,
        ifelse(clase_ternaria == "BAJA+2", 975000, -25000),
        0))]

    # escalo la ganancia como si fuera todo el dataset (misma logica que z290)
    ganancia_test_normalizada <- ganancia_test / ((100 - PARAM$training_pct) / 100)

    resultados <- rbindlist(list( resultados,
      data.table(arbolito= arbolito, ganancia= ganancia_test_normalizada)
    ))
  }

  resultados
}

### Grid Search

In [ ]:
# archivo donde se guarda el checkpoint del grid search (que combinaciones ya se corrieron)
archivo_grid <- "gridsearch_localNuevasCombinaciones.txt"

if (file.exists(archivo_grid)) {
  tb_grid <- fread(archivo_grid)
} else {
  tb_grid <- data.table(
    combo_id = integer(),
    feature_fraction = numeric(),
    cp = numeric(),
    minsplit = integer(),
    minbucket = integer(),
    maxdepth = integer(),
    arbolito = integer(),
    ganancia = numeric()
  )
}

combo_id <- 0

for (v_feature_fraction in PARAM$grid$feature_fraction) {
for (v_minsplit in PARAM$grid$minsplit) {
for (v_minbucket in PARAM$grid$minbucket) {
for (v_maxdepth in PARAM$grid$maxdepth) {

  if (v_minbucket > v_minsplit) next # combinacion redundante/invalida

  combo_id <- combo_id + 1

  # si esta combinacion ya tiene TODOS los puntos de grabar calculados, la salteo
  puntos_hechos <- tb_grid[
    feature_fraction == v_feature_fraction &
    minsplit == v_minsplit &
    minbucket == v_minbucket &
    maxdepth == v_maxdepth,
    arbolito
  ]
  if (all(PARAM$grabar %in% puntos_hechos)) next

  cat(combo_id, " ")
  flush.console()

  rpart_control <- list(
    cp= PARAM$rpart$cp,
    minsplit= v_minsplit,
    minbucket= v_minbucket,
    maxdepth= v_maxdepth
  )

  resultados <- ArbolesAzarososGanancia(v_feature_fraction, rpart_control)

  tb_grid <- rbindlist(list( tb_grid, data.table(
    combo_id= combo_id,
    feature_fraction= v_feature_fraction,
    cp= PARAM$rpart$cp,
    minsplit= v_minsplit,
    minbucket= v_minbucket,
    maxdepth= v_maxdepth,
    arbolito= resultados$arbolito,
    ganancia= resultados$ganancia
  )))

  # grabo el checkpoint despues de cada combinacion, para poder retomar
  #  exactamente desde aca si se corta la conexion
  fwrite(tb_grid, file= archivo_grid, sep= "\t")
}
}
}
}

1  2  3  4  5  6  7  8  9  10  11  12  

In [1]:
library(parallel)
detectCores()

[1] 2

### Resultado: ranking de combinaciones por ganancia del ensemble completo
Cada combinacion tiene una fila por cada punto de `PARAM$grabar`, para ver como evoluciona la ganancia a medida que se agregan arboles. El ranking final se hace sobre el ultimo punto (ensemble completo).

In [ ]:
# ranking final: ganancia del ensemble COMPLETO (ultimo punto de PARAM$grabar) por combinacion
tb_final <- tb_grid[arbolito == max(PARAM$grabar)]
setorder(tb_final, -ganancia)
tb_final[1:10]

combo_id,feature_fraction,cp,minsplit,minbucket,maxdepth,arbolito,ganancia
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>
8,0.7,-1,100,50,8,8,490416667
12,0.7,-1,50,50,8,8,490416667
2,0.5,-1,100,50,8,8,488750000
6,0.5,-1,50,50,8,8,488750000
3,0.5,-1,100,100,6,8,487916667
10,0.7,-1,100,100,8,8,486750000
9,0.7,-1,100,100,6,8,479250000
7,0.7,-1,100,50,6,8,477833333
11,0.7,-1,50,50,6,8,477833333


In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Fri Aug 21 04:29:44 PM 2026"